# CLG query demo — graph-RAG over US + UK primary law

End-to-end walkthrough of the **Common Legal Graph** retrieval pipeline (Phase 4).

Flow:
1. Verify the Neo4j store is reachable + populated.
2. Pick an embedder + a synthesizer backend.
3. Run a question with `run_query` — get a structured `Answer` (text + citations + caveats + used-candidates).
4. Re-run the same question **step-by-step** (parse → seed → expand → good-law → rerank → synthesise) to inspect each stage.
5. Swap synthesizers (`fake` vs real LLM) on the same reranked context.
6. Score the seed gold set (`data/eval/seed.yaml`) with `run_eval`.

**Prereqs**
- `uv sync --extra clg` (and `--extra anthropic` if you want Claude synthesis; `--extra classifier` for the local sentence-transformers embedder).
- A running Neo4j with the CLG schema applied and at least one ingest pass loaded (see `clg ingest …` + `clg load …` + `clg embed`).
- `.env` with `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, and optionally `VOYAGE_API_KEY` / `ANTHROPIC_API_KEY`.
- Open with the `crimellm` kernel.

Every cell falls back to `fake` backends when keys are missing so the notebook still runs (with stub answers).

In [ ]:
from __future__ import annotations

import json

from crimellm.env import load_env

load_env()  # picks up .env at repo root

from crimellm.clg.config import get_settings
from crimellm.clg.graph import get_store

settings = get_settings()
print('Neo4j URI :', settings.neo4j_uri)
print('Embedding :', settings.embedding_model, f'({settings.embedding_dim}d)')

## 1. Graph health

`verify()` raises if the bolt URI / credentials are wrong. The Cypher count gives a quick population check — empty graphs make later cells return empty Answers.

In [ ]:
store = get_store()
store.verify()

rows = store.run(
    """
    CALL {
      MATCH (p:Provision) RETURN 'Provision' AS label, count(p) AS n
      UNION ALL
      MATCH (c:Case)       RETURN 'Case' AS label, count(c) AS n
      UNION ALL
      MATCH (ch:Chunk)     RETURN 'Chunk' AS label, count(ch) AS n
      UNION ALL
      MATCH ()-[r:CITES]->()      RETURN 'CITES' AS label, count(r) AS n
      UNION ALL
      MATCH ()-[r:INTERPRETS]->() RETURN 'INTERPRETS' AS label, count(r) AS n
    }
    RETURN label, n ORDER BY label
    """
)
for r in rows:
    print(f'{r["label"]:<11} {r["n"]:>8}')

## 2. Pick backends

Defaults (when `backend=None`):
- **Embedder** — voyage if `VOYAGE_API_KEY`, else sentence-transformers if `EMBEDDING_MODEL=sentence-transformers/…`, else openai if `OPENAI_API_KEY`, else `fake`.
- **Synthesizer** — anthropic if `ANTHROPIC_API_KEY`, else ollama if its server is reachable, else `fake`.

Pass explicit names below to pin a backend regardless of env.

In [ ]:
from crimellm.clg.embed.embedder import get_embedder
from crimellm.clg.retrieval.synthesize import get_synthesizer

embedder = get_embedder()  # or get_embedder('voyage') / 'sentence-transformers' / 'fake'
synthesizer = get_synthesizer()  # or get_synthesizer('anthropic') / 'ollama' / 'fake'

print('embedder   :', embedder.name, f'(dim={embedder.dim})')
print('synthesizer:', synthesizer.name)

## 3. End-to-end `run_query`

One call → parse the question, seed by vector search, traverse cited / citing / INTERPRETS edges, run a good-law check on Cases, rerank, synthesise with strict citation discipline.

In [ ]:
from crimellm.clg.retrieval import run_query

QUESTION = 'When does receiving stolen property become a federal offence under US law?'

answer = run_query(
    QUESTION,
    jurisdiction=None,        # let parse_query infer (override with 'US' / 'EW' / 'UK')
    as_of=None,               # today
    seed_k=8,
    top_k=6,
    embedder=embedder,
    synthesizer=synthesizer,
    store=store,
)

print(answer.text)
if answer.caveats:
    print('\nCaveats:')
    for c in answer.caveats:
        print(f'  - {c}')
if answer.citations:
    print('\nCited:')
    for c in answer.citations:
        print(f'  - {c}')

In [ ]:
# Full structured payload (what the `--json` CLI flag emits).
print(json.dumps(answer.to_dict(), default=str, indent=2)[:2000])

## 4. Step-by-step

Each stage is a free function in `crimellm.clg.retrieval`. Useful when debugging an empty / wrong answer — you can see exactly which step lost the relevant context.

In [ ]:
from crimellm.clg.retrieval import (
    check_good_law,
    expand_seeds,
    parse_query,
    rerank,
    seed_from_chunks,
    summary_label,
)

query = parse_query(QUESTION)
print(f'raw          : {query.raw}')
print(f'jurisdiction : {query.jurisdiction}')
print(f'as_of        : {query.as_of}')

In [ ]:
seeds = seed_from_chunks(query, embedder, k=8, store=store)
expansions = expand_seeds(seeds, query=query, store=store)
pooled = seeds + expansions

print(f'seeds      : {len(seeds)}')
print(f'expansions : {len(expansions)}')
print(f'pooled     : {len(pooled)}')
for c in pooled[:5]:
    print(f'  [{c.parent_type:<9}] {c.parent_id}  score={c.score:.3f}  source={c.source}')

In [ ]:
case_ids = [c.parent_id for c in pooled if c.parent_type == 'Case']
flags = check_good_law(case_ids, store=store)

ranked = rerank(pooled, today=query.as_of, good_law=flags, top_k=6)

print(f'good-law flags : {sum(len(v) for v in flags.values())} (across {len(flags)} cases)')
print(f'ranked top-{len(ranked)}:')
for i, c in enumerate(ranked, 1):
    badge = ''
    if c.parent_type == 'Case' and c.parent_id in flags:
        badge = f' [{summary_label(flags[c.parent_id])}]'
    print(f'  #{i} [{c.parent_type}] {c.parent_id}{badge}  score={c.score:.3f}')

In [ ]:
manual_answer = synthesizer.synthesise(query=query, candidates=ranked, good_law=flags)
print(manual_answer.text)

## 5. Synthesizer A/B on the same context

Reusing `ranked` from the previous cell lets you compare backends without re-running retrieval. `FakeSynthesizer` always works — useful as a smoke baseline.

In [ ]:
from crimellm.clg.retrieval.synthesize import FakeSynthesizer

fake_answer = FakeSynthesizer().synthesise(query=query, candidates=ranked, good_law=flags)
print('--- fake ---')
print(fake_answer.text[:600])
print()
print(f'--- {synthesizer.name} ---')
print(manual_answer.text[:600])

## 6. Score the seed gold set

`run_eval` calls `run_query` on every question and scores citation recall + good-law correctness. Cheap with `fake` synthesizer; budget LLM calls when running against the real backend.

In [ ]:
from crimellm.clg.eval import load_gold_set
from crimellm.clg.eval.runner import run_eval

gold = load_gold_set('data/eval/seed.yaml')
report = run_eval(gold, embedder=embedder, synthesizer=FakeSynthesizer(), store=store, keep_answers=False)

agg = report.aggregate
print(f'gold      : {report.gold_set_name} v{report.gold_set_version}  ({report.n_questions()} questions)')
print(f'embedder  : {report.embedder_name}')
print(f'synth     : {report.synthesizer_name}')
print(f'aggregate : {agg}')

## Next steps
- Swap `FakeSynthesizer()` for `get_synthesizer('anthropic')` (or `'ollama'`) above and re-run §6 to see real grounded answers + the citation guard at work.
- Tighten retrieval by lowering `seed_k` / `top_k`, or weight `RerankWeights` (see `crimellm.clg.retrieval.rerank`).
- For the citation-treatment side of the graph see the CLI: `clg link citations`, `clg link distill`, `clg link treatment`.